# Patterns 2 & 3: Sequential and Parallel Multi-Agent Systems

> **Google Cloud Pattern Reference**:
> - [Sequential Pattern](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#sequential-pattern)
> - [Parallel Pattern](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#parallel-pattern)

Both patterns use **predefined workflows** — no AI model is needed for orchestration.

## Sequential Pattern
- Agents execute in a **fixed linear order**
- Output of Agent N → Input of Agent N+1
- Use for: ETL pipelines, content generation pipelines, structured business processes
- ✅ Predictable, low overhead | ⚠️ No dynamic re-routing

## Parallel Pattern
- Multiple agents execute **concurrently**
- Outputs are gathered and synthesized
- Use for: Multi-perspective analysis, fan-out data gathering, independent sub-tasks
- ✅ Lower latency | ⚠️ Higher immediate cost, complex aggregation

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

## Part A: Sequential Pattern

```
User Request
     │
     ▼
[Agent 1: Research] ──► [Agent 2: Analysis] ──► [Agent 3: Report Writer]
```

### A1. Sequential Pattern — Plain Python pipeline

The sequential pattern is just chained `agent.run()` calls in Python.
No framework class needed — each agent's output feeds the next.

> **Note on web search tools**: `DuckDuckGoTools` relies on a web scraper (`ddgs`) that
> breaks whenever DuckDuckGo changes its page structure. For reliable course demos,
> this pipeline uses the model's built-in knowledge instead.
> To add live search: install a stable API-backed search tool (e.g. Tavily, SerpAPI)
> and swap it in as `tools=[TavilyTools()]`.

In [15]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

class NewsAnalysisPipeline:
    """
    Sequential Pattern: Research → Analysis → Report
    Each agent's output feeds the next in a fixed linear order.
    No AI model is used for orchestration — the pipeline is hardcoded Python.
    """

    def __init__(self):
        model = OpenAIChat(id="google/gemini-2.5-flash")

        # Agent 1: Gather knowledge on the topic
        self.researcher = Agent(
            name="Researcher",
            model=model,
            instructions=[
                "Summarize the 5 most important recent developments about the given topic.",
                "Include key players, trends, and notable announcements.",
                "Use your training knowledge — be specific and factual.",
            ],
            markdown=False,
        )

        # Agent 2: Analyze and extract insights
        self.analyst = Agent(
            name="Analyst",
            model=model,
            instructions=[
                "Analyze the research provided.",
                "Identify key trends, risks, and opportunities.",
                "Rate overall sentiment: Positive / Neutral / Negative.",
            ],
            markdown=False,
        )

        # Agent 3: Write the final report
        self.report_writer = Agent(
            name="Report Writer",
            model=model,
            instructions=[
                "Write a concise executive report from the analysis.",
                "Include: Executive Summary, Key Findings, Sentiment, Recommendations.",
                "Format in clean Markdown with headers.",
            ],
            markdown=True,
        )

    def run(self, topic: str) -> str:
        print(f"\n🔍 Step 1: Researching '{topic}'...")
        research = self.researcher.run(f"Research: {topic}")

        print("📊 Step 2: Analyzing findings...")
        analysis = self.analyst.run(
            f"Analyze this research:\n{research.content}"
        )

        print("📝 Step 3: Writing report...")
        report = self.report_writer.run(
            f"Write a report based on this analysis:\n{analysis.content}"
        )

        return report.content


pipeline = NewsAnalysisPipeline()
result = pipeline.run("Generative AI agents 2025")
print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(result)


🔍 Step 1: Researching 'Generative AI agents 2025'...
📊 Step 2: Analyzing findings...
📝 Step 3: Writing report...

FINAL REPORT
## Executive Report: Generative AI Agents in 2025

### Executive Summary

By 2025, Generative AI agents are poised for a transformative evolution, shifting from simple content generation to highly autonomous, reasoning, and real-world interactive systems. These "agentic" AIs will specialize in complex tasks, operate across multiple modalities (text, image, audio, video), and actively collaborate with humans and other agents. While offering immense potential for productivity, new discoveries, and significant ROI, this advancement is shadowed by critical challenges including job displacement, heightened ethical dilemmas, and complex accountability issues. The overall sentiment is one of **optimistic caution**, recognizing the profound opportunities while demanding robust ethical and safety frameworks for responsible development and deployment.

### Key Findings


### A2. Sequential Pattern using Agno Team (`coordinate` mode)

> **Agno 2.x Team modes**: `coordinate` | `route` | `broadcast` | `tasks`
> There is no `sequential` mode. For a pure code-driven pipeline with zero LLM overhead,
> use the plain-Python class shown in A1.
>
> `coordinate` mode is the closest equivalent: the team leader LLM decides which members
> to call and how to synthesize results. Pass `model=` on the Team — it's required for coordination.

In [16]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat

model = OpenAIChat(id="google/gemini-2.5-flash")

data_extractor = Agent(
    name="Data Extractor",
    role="Extract structured data from raw text",
    model=model,
    instructions=["Extract key entities: companies, dates, numbers, locations."],
)

data_cleaner = Agent(
    name="Data Cleaner",
    role="Clean and standardize extracted data",
    model=model,
    instructions=[
        "Normalize dates to ISO format (YYYY-MM-DD).",
        "Standardize company names to their official names.",
        "Flag any ambiguous or missing data.",
    ],
)

data_formatter = Agent(
    name="Data Formatter",
    role="Format clean data into a markdown table",
    model=model,
    instructions=["Output a clean markdown table with the structured data."],
    markdown=True,
)

# mode='coordinate': team leader LLM orchestrates which members to call.
# model= is required when using Team — it drives the coordinator.
etl_team = Team(
    name="ETL Pipeline",
    model=model,           # Required coordinator model
    mode="coordinate",     # 'coordinate' is the Agno 2.x mode for multi-step coordination
    members=[data_extractor, data_cleaner, data_formatter],
    instructions=[
        "Process input through Extract → Clean → Format in strict sequence.",
        "Each agent's output becomes input for the next.",
        "Return the final formatted markdown table.",
    ],
    markdown=True,
)

raw_text = """
Apple Inc. announced on jan 15, 2025 that it will acquire startup XYZ for $2.5 billion.
The deal is expected to close by Q2 2025. microsoft also confirmed a $1B investment 
in OpenAI on 12/20/2024. google LLC reported quarterly revenue of $89.9B for Q4 2024.
"""

etl_team.print_response(
    f"Process this raw business news text through the ETL pipeline:\n{raw_text}",
    stream=True,
)

Output()

---

## Part B: Parallel Pattern

```
                   ┌─► [Sentiment Agent] ──┐
                   │                        │
User Request ──────┼─► [Keyword Agent]   ──┼──► [Synthesizer] ──► Final Output
                   │                        │
                   └─► [Category Agent]  ──┘
```

### B1. Parallel Pattern using Agno Team (`broadcast` mode)

> **Agno 2.x**: Use `mode='broadcast'` for parallel fan-out — the team delegates the **same task** to **all members simultaneously**,
> then synthesizes their responses. `model=` is required for the synthesizer step.

In [17]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat

model = OpenAIChat(id="google/gemini-2.5-flash")

# 4 specialized agents — each analyzes the feedback from a different angle
sentiment_agent = Agent(
    name="Sentiment Analyst",
    role="Analyze customer feedback sentiment",
    model=model,
    instructions=[
        "Rate sentiment: Very Positive / Positive / Neutral / Negative / Very Negative.",
        "Provide a sentiment score from -1.0 (most negative) to +1.0 (most positive).",
    ],
)

keyword_agent = Agent(
    name="Keyword Extractor",
    role="Extract key topics and phrases from customer feedback",
    model=model,
    instructions=[
        "Extract the top 5-7 keywords/phrases.",
        "Group them by theme (product, service, price, experience).",
    ],
)

category_agent = Agent(
    name="Category Classifier",
    role="Classify feedback into support categories",
    model=model,
    instructions=[
        "Categorize feedback: Bug Report / Feature Request / Complaint / Compliment / General Inquiry.",
        "Identify the primary product/feature being discussed.",
    ],
)

urgency_agent = Agent(
    name="Urgency Detector",
    role="Assess urgency and priority of customer feedback",
    model=model,
    instructions=[
        "Rate urgency: Critical / High / Medium / Low.",
        "Flag if customer is threatening churn or legal action.",
        "Recommend response time (e.g., within 1 hour, 24 hours, 1 week).",
    ],
)

# mode='broadcast': all members receive the same task and run in parallel.
# The team leader model then synthesizes all member responses.
feedback_analysis_team = Team(
    name="Customer Feedback Analyzer",
    model=model,          # Required: synthesizer model
    mode="broadcast",     # Parallel fan-out: all agents run simultaneously
    members=[sentiment_agent, keyword_agent, category_agent, urgency_agent],
    instructions=[
        "Each member agent analyzes the feedback independently.",
        "Synthesize all analyses into a comprehensive feedback report with clear sections.",
    ],
    markdown=True,
)

customer_feedback = """
I've been using your product for 3 months and I'm extremely frustrated. 
The app crashes every time I try to export a report, which is critical for my business. 
I've submitted 3 support tickets and got no response. If this isn't fixed in the next 
48 hours, I'm switching to a competitor and will be requesting a refund for the entire year.
The features themselves are great, but the reliability is unacceptable.
"""

feedback_analysis_team.print_response(
    f"Analyze this customer feedback:\n{customer_feedback}",
    stream=True,
)

Output()

### B2. Parallel Pattern using asyncio for fine-grained control
For maximum control over parallel execution — no team overhead, pure Python concurrency.

In [18]:
import asyncio
from agno.agent import Agent
from agno.models.openai import OpenAIChat

async def run_parallel_analysis(topic: str):
    """Fan out topic to 3 specialized agents in parallel, then synthesize."""

    model = OpenAIChat(id="google/gemini-2.5-flash")

    market_agent = Agent(
        name="Market Analyst",
        model=model,
        instructions=["Analyze market opportunity and size."],
    )
    tech_agent = Agent(
        name="Tech Analyst",
        model=model,
        instructions=["Analyze technical feasibility and stack requirements."],
    )
    risk_agent = Agent(
        name="Risk Analyst",
        model=model,
        instructions=["Identify top 5 risks and mitigation strategies."],
    )

    # Run all 3 agents concurrently
    results = await asyncio.gather(
        market_agent.arun(f"Market analysis for: {topic}"),
        tech_agent.arun(f"Tech analysis for: {topic}"),
        risk_agent.arun(f"Risk analysis for: {topic}"),
    )

    # Synthesize
    synthesizer = Agent(
        name="Synthesizer",
        model=model,
        instructions=["Combine all analyses into an executive summary. Use markdown headers."],
        markdown=True,
    )

    combined = "\n\n".join([
        f"## Market Analysis\n{results[0].content}",
        f"## Tech Analysis\n{results[1].content}",
        f"## Risk Analysis\n{results[2].content}",
    ])

    final = await synthesizer.arun(f"Synthesize these analyses for '{topic}':\n{combined}")
    return final.content


result = await run_parallel_analysis("AI-powered legal document review SaaS")
print(result)

## Executive Summary: AI-powered Legal Document Review SaaS

This executive summary synthesizes the market, technical, and risk analyses for an AI-powered Legal Document Review SaaS, highlighting the core opportunity, technical feasibility, and critical challenges to navigate.

### Market Opportunity: High Growth & Substantial Demand

The market for AI-powered legal document review SaaS is **robust, rapidly growing, and presents a significant, multi-billion dollar opportunity.** Driven by the exponential growth of electronic data, escalating legal costs, and the need for efficiency, legal professionals are increasingly adopting these solutions.

**Key Market Drivers:**
*   **Explosion of Data:** Traditional manual review cannot cope with the sheer volume of ESI.
*   **Cost Reduction Pressure:** AI offers significant savings in human review hours.
*   **Need for Speed & Efficiency:** Faster review cycles for time-sensitive legal processes.
*   **AI Advancements:** Improved NLP, ML, and 

### Comparison Summary

| Dimension | Sequential | Parallel |
|-----------|-----------|----------|
| Execution | Linear, ordered | Concurrent |
| Latency | Higher (sum of steps) | Lower (max of steps) |
| Cost | Lower (1 agent at a time) | Higher (all agents at once) |
| Orchestration | Hardcoded code | Hardcoded code |
| Best for | ETL pipelines, content pipelines | Multi-perspective analysis, data gathering |
| Plain Python | Plain class (A1) | `asyncio.gather` (B2) |
| Agno Team | `Team(mode='coordinate', model=...)` | `Team(mode='broadcast', model=...)` |

> **Agno 2.x Team modes**: `coordinate` (LLM picks/routes members) · `route` (LLM picks one) · `broadcast` (all members, parallel) · `tasks` (autonomous task queue)

👉 **Next**: `5_react_agent.py` — the ReAct (Reason and Act) pattern.